# 분자 최적화 — 동적 계획법 (Dynamic Programming, DP)

## 학습 목표
- 지난주 **탐욕 알고리즘**(greedy algorithm)은 매 순간 가장 좋은 이웃 하나만 선택했다 → **지역 최적해**(local optimum)에 갇힐 수 있다.
- 이번 주 **동적 계획법** (dynamic programming, DP) — 모든 경로를 시도하되, 같은 계산은 **저장해서 재사용**(caching).
- 분자 공간이 너무 커지면 **KISTI 슈퍼컴퓨터 + mpi4py**로 병렬화.

| 알고리즘 (한국어) | 영어 | 전략 | 결과 |
|---|---|---|---|
| 탐욕 | greedy | 한 길만 | 지역 최적해 (local optimum) |
| 동적 계획법 | dynamic programming | 모든 길 + 결과 저장 | 전역 최적해 (global optimum) |


## 1. 지난주 복습 — 탐욕(greedy)의 한계

매 단계에서 점수가 가장 높은 이웃 **1개만** 선택한다.

```mermaid
graph TD
    M["시작 분자 (점수 -0.7)"]
    M -->|버림| A["이웃 A (점수 -0.5)"]
    M ==>|선택| B["이웃 B (점수 -0.1)"]
    M -->|버림| C["이웃 C (점수 -0.3)"]
    B ==> Next["B의 이웃 중 또 1개만..."]
    style B fill:#ffeb3b
    style Next fill:#ffeb3b
```

**문제**: B가 지금은 좋아 보여도 두 단계 뒤에 막힐 수 있다. A에서 출발했다면 더 좋았을 수도 있다. 한 번 선택하면 되돌릴 수 없다 → **지역 최적해(local optimum)** 함정.


## 2. 모든 경로를 시도하는 DP

DP는 **모든 이웃을 끝까지** 시도한 뒤 그 중 최고를 고른다.

```mermaid
graph TD
    M[시작 분자]
    M --> A[이웃 A]
    M --> B[이웃 B]
    M --> C[이웃 C]
    A --> A1["A의 이웃들..."]
    B --> B1["B의 이웃들..."]
    C --> C1["C의 이웃들..."]
    A1 --> Best["전부 비교 후<br/>전역 최적 선택"]
    B1 --> Best
    C1 --> Best
    style Best fill:#90ee90
```

이렇게 모든 경로를 보면 **전역 최적해(global optimum)**를 찾을 수 있다. 단점: 계산이 너무 많다 → 같은 계산이 반복되는 문제 발생 → **캐싱(caching)**으로 해결.


## 3. 캐싱(caching)이란?

**한 번 계산한 결과를 저장해두고, 같은 계산이 또 필요할 때 다시 꺼내 쓰는 기법.**

- 저장소를 **캐시(cache)**라 부른다 — 파이썬에서는 보통 `dict`(사전)을 캐시로 쓴다.
- 결과를 사전에 넣어두는 행위를 **캐싱(caching)**이라 한다.

아래는 일부러 느린 함수에 캐시를 붙여 빠르게 만드는 예시.


In [1]:
import time

def slow_double(x):
    time.sleep(0.5)               # 일부러 0.5초 걸리게
    return x * 2


# === 캐시 없이 ===
t = time.time()
slow_double(7)
slow_double(7)                    # 같은 값인데도 또 0.5초!
print(f'캐시 없음: {time.time()-t:.2f}초')


# === 캐시 있게 ===
cache = {}

def fast_double(x):
    if x in cache:                # 사전에 있으면 바로 꺼냄
        return cache[x]
    result = slow_double(x)       # 없으면 계산
    cache[x] = result             # 사전에 저장
    return result

t = time.time()
fast_double(7)                    # 첫 호출 — 0.5초
fast_double(7)                    # 두 번째 — 즉시 (캐시 hit)
print(f'캐시 있음: {time.time()-t:.2f}초')

# 예상 출력:
# 캐시 없음: 1.00초
# 캐시 있음: 0.50초


캐시 없음: 1.00초
캐시 있음: 0.50초


## 4. 같은 분자가 여러 경로로 도달한다

분자 DP에서 캐싱이 중요한 이유: 서로 다른 변형 경로가 **같은 분자**에 도달할 수 있다.

```mermaid
graph TD
    M[시작 m]
    M --> a[분자 a]
    M --> b[분자 b]
    a --> X[같은 분자 X]
    b --> X
    X --> next["X에서 더 변형해본다..."]
    style X fill:#ff9999
```

**X에서 출발했을 때의 최적 점수는 한 번만 계산하면 충분.** 두 번째 도달했을 때는 캐시에서 꺼내쓴다 → 큰 시간 절약.

이렇게 **같은 부분 문제(subproblem)가 여러 번 등장하는** 구조에서 DP가 빛난다.


## 5. 분자 DP 코드

지난주와 같은 점수 함수 `score`와 이웃 함수 `neighbors`를 사용한다.


In [2]:
from rdkit import Chem, RDLogger
from rdkit.Chem import QED, Crippen
RDLogger.DisableLog('rdApp.*')


def score(smi):
    mol = Chem.MolFromSmiles(smi)
    logp = min(Crippen.MolLogP(mol), 5) #5보다 작으면 logP 계산 값을 사용하고, 5보다 큰 값이 나오면 5로 대체.
    return QED.qed(mol) - 0.5 * logp


def neighbors(smi):
    mol = Chem.MolFromSmiles(smi)
    result = []
    for i in range(mol.GetNumAtoms()):
        for atom_num in [6, 7, 8]:        # C=6, N=7, O=8
            rw = Chem.RWMol(mol)
            rw.GetAtomWithIdx(i).SetAtomicNum(atom_num)
            try:
                Chem.SanitizeMol(rw)
                result.append(Chem.MolToSmiles(rw))
            except:
                pass
    return result


## 6. 재귀 함수 (recursive function)란?

**자기 자신을 호출하는 함수.** 큰 문제를 같은 모양의 더 작은 문제로 쪼개서 푼다.

재귀 함수가 성립하려면 두 가지가 필요:
1. **베이스 케이스 (base case)** — 더 작아질 수 없는 상황. 여기서 멈춰야 한다 (없으면 무한 호출!)
2. **재귀 케이스 (recursive case)** — 자기 자신을 더 작은 입력으로 호출

가장 단순한 예: 팩토리얼 `n! = n × (n-1) × ... × 1`

```python
def factorial(n):
    if n == 0:                       # 베이스 케이스 — 여기서 멈춤
        return 1
    return n * factorial(n - 1)      # 재귀 케이스 — 자기 자신 호출
```

### 동작 방식 — `factorial(3)` 호출

함수가 한 번 호출되면, 더 작은 입력으로 자기 자신을 부르며 **점점 깊이 들어간다**. 베이스 케이스에 도달하면 **거꾸로 거슬러 올라오며** 결과를 합쳐 반환한다.

```mermaid
sequenceDiagram
    participant Main
    participant F3 as factorial(3)
    participant F2 as factorial(2)
    participant F1 as factorial(1)
    participant F0 as factorial(0)
    Main->>F3: 호출
    F3->>F2: factorial(2) 호출 (작아짐 ↓)
    F2->>F1: factorial(1) 호출 (작아짐 ↓)
    F1->>F0: factorial(0) 호출 (작아짐 ↓)
    Note over F0: 베이스 케이스!<br/>1 반환
    F0-->>F1: return 1
    Note over F1: 1 × 1 = 1
    F1-->>F2: return 1
    Note over F2: 2 × 1 = 2
    F2-->>F3: return 2
    Note over F3: 3 × 2 = 6
    F3-->>Main: return 6
```

**핵심**: 들어갈 때(↓)는 문제를 작게 쪼개고, 나올 때(↑)는 결과를 합친다.

> 다음에 만들 `best_score(smi, k)`도 재귀 함수다. `(smi, k-1)`로 더 작게 자기 자신을 호출하고, 베이스 케이스는 `k = 0`. 거기에 **캐싱**까지 더해진 것이 DP다.


In [ ]:
def factorial(n):
    if n == 0:                       # 베이스 케이스
        return 1
    return n * factorial(n - 1)      # 재귀 케이스


print('factorial(0) =', factorial(0))
print('factorial(3) =', factorial(3))
print('factorial(5) =', factorial(5))

# 예상 출력:
# factorial(0) = 1
# factorial(3) = 6
# factorial(5) = 120


## 7. DP 함수 만들기

`best_score(smi, k)` = 분자 `smi`에서 시작해 **최대 k번 변형**하여 도달할 수 있는 **최고 점수**.

규칙:
- `k = 0`이면 더 변형 못 함 → 현재 점수 반환
- `k > 0`이면 (1) 지금 멈춘 점수와 (2) 모든 이웃에서 `k-1`번 더 변형한 점수 중 **최고**를 선택
- 같은 `(smi, k)`를 다시 만나면 → 캐시에서 꺼내씀

### 캐시 사전(dict)의 구조

`memo`는 파이썬 사전(dictionary)이다 — **키(key)** 로 즉시 검색하고 **값(value)** 을 꺼낼 수 있다.

```python
memo = {}                                # 빈 사전으로 시작
memo[(smi, k)] = best                    # 저장: 키 (smi, k) → 값 best
result = memo[(smi, k)]                  # 검색: 키로 즉시 꺼냄
```

#### 왜 키가 `(smi, k)` 두 개의 묶음인가?

`best_score`의 답은 **분자 `smi`와 남은 횟수 `k` 둘 다**에 따라 달라진다. 같은 분자라도 k가 다르면 답이 다르다.

| 호출 | 의미 | 답(예시) |
|---|---|---|
| `best_score('X', 0)` | X에서 더 변형 못 함 | `score(X)` = 0.10 |
| `best_score('X', 1)` | X에서 1번 더 변형 가능 | 0.25 |
| `best_score('X', 2)` | X에서 2번 더 변형 가능 | 0.40 |

→ **두 정보(`smi`, `k`)를 모두 가져야 부분 문제(subproblem)를 식별**할 수 있다. 그래서 두 값을 묶은 **튜플 `(smi, k)`를 키로** 쓴다.

#### 값(value)은 무엇인가?

그 부분 문제의 **답 — 즉 최고 점수 `best`**. 한 번 계산했으면 답을 저장해두고, 다음에 같은 `(smi, k)`로 호출되면 재계산 없이 꺼내쓴다.

#### memo 안의 모습

각 부분 문제는 사전에서 한 줄(엔트리)을 차지한다.

```mermaid
graph LR
    subgraph calls["호출들 (부분 문제)"]
        C1["best_score('CCCC', 0)"]
        C2["best_score('CCCC', 1)"]
        C3["best_score('CCCN', 1)"]
        C4["best_score('CCNN', 0)"]
    end
    subgraph dict["memo (dict)"]
        E1["키 ('CCCC', 0)<br/>값 -0.42"]
        E2["키 ('CCCC', 1)<br/>값 -0.10"]
        E3["키 ('CCCN', 1)<br/>값  0.05"]
        E4["키 ('CCNN', 0)<br/>값  0.20"]
    end
    C1 -->|저장| E1
    C2 -->|저장| E2
    C3 -->|저장| E3
    C4 -->|저장| E4
```

같은 분자 `'CCCC'`라도 `k=0`과 `k=1`은 **서로 다른 부분 문제**라 따로 저장되는 점에 주목.

### 함수 동작 흐름

```mermaid
flowchart TD
    Start(["best_score(smi, k) 호출"])
    Start --> Cache{"(smi, k)가<br/>memo에 있나?"}
    Cache -->|있다 ✓| Hit["memo에서 꺼내 반환<br/>(끝)"]
    Cache -->|없다| Init["best = score(smi)<br/>(지금 멈춘 점수)"]
    Init --> Check{"k가 0보다 큰가?"}
    Check -->|예| Loop["각 이웃 n에 대해<br/>cand = best_score(n, k-1)<br/>best = max(best, cand)"]
    Loop --> Save
    Check -->|아니오| Save["memo[(smi, k)] = best<br/>(캐시에 저장)"]
    Save --> Done(["best 반환"])

    style Cache fill:#bbdefb
    style Hit fill:#c8e6c9
    style Save fill:#ffe0b2
```

### 실제 호출 예시 — K=2

시작 분자 `M`에서 `best_score(M, 2)`를 호출하면 함수가 재귀적으로 펼쳐진다 (이웃이 a, b 두 개라고 가정).

```mermaid
graph TD
    Root["best_score(M, 2)"]
    Root --> A1["best_score(a, 1)"]
    Root --> B1["best_score(b, 1)"]

    A1 --> AA["best_score(a₁, 0)<br/>점수 계산"]
    A1 --> AB["best_score(a₂, 0)<br/>점수 계산"]
    A1 --> AX["best_score(X, 0)<br/>점수 계산 후 캐싱"]

    B1 --> BA["best_score(b₁, 0)<br/>점수 계산"]
    B1 --> BB["best_score(b₂, 0)<br/>점수 계산"]
    B1 --> BX["best_score(X, 0)<br/>✓ 캐시 hit, 즉시 반환"]

    style AX fill:#ffe0b2
    style BX fill:#a5d6a7
```

**핵심 관찰**: 같은 분자 `X`가 두 경로(`M → a → X`, `M → b → X`)로 도달. 첫 호출 때 캐시에 저장 → 두 번째는 즉시 꺼냄. K가 커질수록 이런 중복이 폭발적으로 늘어나서 캐싱의 효과가 커진다.


In [3]:
memo = {}    # 캐시: (분자, 남은 횟수) → 최고 점수

def best_score(smi, k):
    if (smi, k) in memo:                    # 캐시 hit
        return memo[(smi, k)]

    best = score(smi)                       # 지금 멈췄을 때 점수
    if k > 0:
        for n in neighbors(smi):
            cand = best_score(n, k - 1)
            if cand > best:
                best = cand

    memo[(smi, k)] = best                   # 결과 캐싱
    return best


## 8. 실행 — 이부프로펜에서 출발

In [4]:
import time

memo.clear()
start = 'CC(C)Cc1ccc(cc1)C(C)C(=O)O'    # 이부프로펜
K = 2

t = time.time()
best = best_score(start, K)
print(f'시작 점수 : {round(score(start), 3)}')
print(f'DP 최고점 : {round(best, 3)}  (K={K})')
print(f'캐시 크기 : {len(memo)}')
print(f'시간      : {time.time()-t:.2f}초')

# 예상 출력:
# 시작 점수 : -0.715
# DP 최고점 : 0.353  (K=2)
# 캐시 크기 : 약 120
# 시간      : 0.1 ~ 0.3초


시작 점수 : -0.715
DP 최고점 : 0.353  (K=2)
캐시 크기 : 123
시간      : 0.31초


## 9. 탐욕(greedy) vs DP — 같은 K에서 비교

In [5]:
def greedy(smi, max_steps):
    cur = smi
    for _ in range(max_steps):
        nxt = max(neighbors(cur), key=score)
        if score(nxt) <= score(cur):
            break
        cur = nxt
    return score(cur)

memo.clear()
print(f'탐욕 K=2 : {round(greedy(start, 2), 3)}')
print(f'DP   K=2 : {round(best_score(start, 2), 3)}')

# 예상 출력 (이 좁은 탐색 공간에서는 탐욕도 운 좋게 최적해를 찾음):
# 탐욕 K=2 : 0.353
# DP   K=2 : 0.353
#
# 탐색 공간이 커지면 (다음 셀) 탐욕은 지역 최적해에 갇히고 DP가 이긴다.


탐욕 K=2 : 0.347
DP   K=2 : 0.353


## 10. 복잡도 폭발 — 왜 KISTI가 필요한가

지금까지의 `neighbors`는 **치환만** 해서 분자 크기가 고정. 도달 가능한 분자 수가 작아 K=3까지 1초 안에 끝난다.

현실의 분자 설계는 **원자 추가/삭제**도 한다. 이웃을 확장하면 상태 공간이 폭발한다.


In [6]:
def neighbors_big(smi):
    """치환 + 각 원자에 C/N/O 하나씩 붙이기"""
    mol = Chem.MolFromSmiles(smi)
    result = list(neighbors(smi)) #원자 치환한 smiles 저장.

    #원자 추가한 smiles 저장.
    for i in range(mol.GetNumAtoms()):
        for an in [6, 7, 8]:
            rw = Chem.RWMol(mol)
            new_idx = rw.AddAtom(Chem.Atom(an))
            rw.AddBond(i, new_idx, Chem.BondType.SINGLE)
            try:
                Chem.SanitizeMol(rw)
                result.append(Chem.MolToSmiles(rw))
            except:
                pass
    return result


memo_big = {}
def best_score_big(smi, k):
    if (smi, k) in memo_big:
        return memo_big[(smi, k)]
    best = score(smi)
    if k > 0:
        for n in neighbors_big(smi):
            cand = best_score_big(n, k - 1)
            if cand > best:
                best = cand
    memo_big[(smi, k)] = best
    return best


for K in [1, 2, 3]:
    memo_big.clear()
    t = time.time()
    s = best_score_big(start, K)
    print(f'K={K}: 점수 {round(s,3):>6}, 캐시 {len(memo_big):>6}, {time.time()-t:>6.2f}초')

# 예상 출력 (대략, PC에 따라 다름):
# K=1: 점수 -0.092, 캐시     40,   0.04초
# K=2: 점수  0.353, 캐시    873,   1.0초
# K=3: 점수  0.742, 캐시  13853,  30~60초
#
# K가 1 늘 때마다 시간이 ~30배 → K=4는 수십 분, K=5는 시간 단위


K=1: 점수 -0.092, 캐시     40,   0.08초
K=2: 점수  0.353, 캐시    873,   1.87초
K=3: 점수  0.742, 캐시  13853,  33.96초


## 11. KISTI에서 mpi4py로 병렬 실행

지금까지의 한 PC DP는 K=3까지가 한계. KISTI 슈퍼컴퓨터에서 같은 알고리즘을 **마스터-워커 패턴**으로 병렬화하면 K=4, K=5도 다룰 수 있다.

### 마스터-워커 동작

```mermaid
sequenceDiagram
    participant M as 마스터 (rank 0)
    participant W1 as 워커 1
    participant W2 as 워커 2
    M->>W1: 분자 묶음 1
    M->>W2: 분자 묶음 2
    Note over W1,W2: 각 워커가 점수 계산
    W1->>M: 결과
    W2->>M: 결과
    Note over M: 합치고 top-N을 다음 단계로
```

각 깊이마다 (1) 마스터가 후보 분자를 모으고 → (2) 워커들에게 분배 → (3) 각자 점수 계산 → (4) 마스터가 모아 top-N 선택. K번 반복.

### 실제 병렬 계산이 일어나는 모습

한 깊이에서 후보 분자 35개가 만들어졌다고 하자. 워커가 4명이라면 마스터는 **각 워커에게 약 9개씩** 나눠준다.

```mermaid
flowchart TD
    F["frontier (현재 깊이의 분자들)"]
    F --> Gen["neighbors() 호출<br/>→ 후보 분자 35개 생성"]
    Gen --> Split{"마스터가 균등 분배"}
    Split -->|"후보 1 ~ 9"| W1
    Split -->|"후보 10 ~ 18"| W2
    Split -->|"후보 19 ~ 27"| W3
    Split -->|"후보 28 ~ 35"| W4

    W1["<b>워커 1</b><br/>for 분자 in 묶음:<br/>　s = score(분자)<br/>(9번 호출)"]
    W2["<b>워커 2</b><br/>for 분자 in 묶음:<br/>　s = score(분자)<br/>(9번 호출)"]
    W3["<b>워커 3</b><br/>for 분자 in 묶음:<br/>　s = score(분자)<br/>(9번 호출)"]
    W4["<b>워커 4</b><br/>for 분자 in 묶음:<br/>　s = score(분자)<br/>(8번 호출)"]

    W1 -->|"(점수, 분자) 9개"| G
    W2 -->|"(점수, 분자) 9개"| G
    W3 -->|"(점수, 분자) 9개"| G
    W4 -->|"(점수, 분자) 8개"| G

    G[<b>마스터</b>: 모든 결과 수합]
    G --> Best[최고점 갱신]
    Best --> Top["점수 정렬 → top-50 선택"]
    Top --> Loop["frontier 갱신 → 다음 깊이로"]

    style W1 fill:#e1f5fe
    style W2 fill:#e1f5fe
    style W3 fill:#e1f5fe
    style W4 fill:#e1f5fe
    style G fill:#fff9c4
```

**핵심 — 워커들은 서로 대화하지 않는다**:
- 각 워커는 **자기 묶음의 분자들에 대해서만** `score()` 호출 (독립 작업)
- 4 워커가 **동시에(병렬로)** 자기 일을 수행 → 9번 호출하는 시간 ≈ 35번을 한 PC가 혼자 하는 시간의 1/4
- 한 PC가 순차 실행한다면 35번 모두 직렬 → 4 워커는 약 **4배 빠름** (워커 수에 비례)

KISTI에서 **32 워커**를 쓰면 약 32배 속도 향상. 한 깊이에서 1000개 후보가 나와도 한 워커당 ~30번이면 끝난다 → K=4, K=5도 실용 시간에 가능.

### 시리얼 vs 병렬 — 시간이 어떻게 줄어드나

같은 35개 분자에 대해 `score()` 한 번 계산이 1초 걸린다고 가정.

- **시리얼 (serial, 1 CPU)**: 분자 35개를 한 번에 한 개씩 순서대로 처리 → **총 35초**
- **병렬 (parallel, 4 워커)**: 4명이 동시에 자기 묶음을 처리 → **약 9초**

```mermaid
gantt
    title 35개 분자 점수 계산 — 시리얼 vs 병렬
    dateFormat X
    axisFormat %s초

    section 시리얼 (1 CPU)
    분자 1~9 처리    :s1, 0, 9
    분자 10~18 처리  :s2, 9, 9
    분자 19~27 처리  :s3, 18, 9
    분자 28~35 처리  :s4, 27, 8

    section 병렬 워커 1
    분자 1~9   :p1, 0, 9
    section 병렬 워커 2
    분자 10~18 :p2, 0, 9
    section 병렬 워커 3
    분자 19~27 :p3, 0, 9
    section 병렬 워커 4
    분자 28~35 :p4, 0, 8
```

차이를 한 줄로:

| 방식 | 처리 순서 | 총 시간 |
|---|---|---|
| 시리얼 (1 CPU) | 35개를 **차례대로** 한 줄로 | 35 × score 시간 |
| 병렬 (N 워커) | 35개를 **N 묶음으로 쪼개 동시에** | (35 / N) × score 시간 |

**이상적 속도 향상 = 워커 수**. KISTI 32 워커면 약 32배 빠름 (실제로는 마스터-워커 통신 비용이 있어 약간 못 미침).

### 제출할 파일

같은 폴더의 **`w10-1_molecule_dp_kisti.py`** — 7주차 마스터-워커 패턴(`w7-2_mpi4py_code_explained.py`)과 동일한 구조.

### 실행 명령 — 2단계 진행

**1단계: 개인 PC에서 가볍게 테스트** (KISTI 제출 전 동작 확인)

스크립트 상단의 `TEST_MODE = True` 로 두고:

```bash
mpirun -np 4 python w10-1_molecule_dp_kisti.py
```

- `TEST_MODE = True`이면 `K = 2`, `BEAM = 10`으로 작게 돌아감 → 수 초 내 완료
- 마스터-워커 출력이 정상으로 찍히는지, 점수가 갱신되는지 먼저 확인

**2단계: KISTI 본 실행**

테스트가 잘 돌아가면 스크립트 상단의 `TEST_MODE = False` 로 바꾸고 (자동으로 `K = 4`, `BEAM = 50`):

```bash
mpirun -np 32 python w10-1_molecule_dp_kisti.py
```

- `-np 32` → 1 마스터 + 31 워커
- KISTI에서는 작업 스케줄러(Slurm 등)에 제출 — 4주차 KISTI 강의 참고

### 예상 출력 (요약)

```
[마스터] 워커 31명, 시작 분자: CC(C)Cc1ccc(cc1)C(C)C(=O)O
[마스터] K=4, BEAM=50
============================================================
[마스터] 깊이 1: 평가할 후보 30~50개
[마스터] 깊이 2: 평가할 후보 800~900개
  [마스터] 새 최적: 점수 0.353, ...
[마스터] 깊이 3: 평가할 후보 10000+
  [마스터] 새 최적: 점수 0.742, ...
[마스터] 깊이 4: ...
  [마스터] 새 최적: 점수 ...
============================================================
[마스터] 최적 점수 : 1.0+
[마스터] 최적 분자 : (변형된 SMILES)
[마스터] 총 시간   : XX초
```

### PC vs KISTI 시간 비교 (대략)

| K | PC (1 코어) | KISTI (32 코어) |
|---|---|---|
| 2 | ~1초 | ~0.1초 |
| 3 | 30~60초 | ~수 초 |
| 4 | 수십 분 | ~분 단위 |
| 5 | 시간 단위 | 수십 분 |

병렬화로 더 깊은 K → 더 좋은 분자에 도달할 수 있다.

---
## 오늘 배운 것

- **재귀 함수 (recursive function)** — 자기 자신을 더 작은 입력으로 호출. 베이스 케이스에서 멈춤
- **캐싱 (caching)** — 한 번 계산한 결과를 사전(dict)에 저장하고 재사용
- **동적 계획법 (dynamic programming, DP)** = 재귀 + 캐싱
- **분자 DP**: `best_score(smi, k)`로 모든 변형 경로의 전역 최적해 탐색
- **확장**: 탐색 공간이 폭발하면 KISTI mpi4py로 병렬화 (`mpirun -np 32 python w10-1_molecule_dp_kisti.py`)